# Paper Figures: When Not to Estimate

4-page paper figure generation. 3 configs: ELAA 15GHz, ELAA 28GHz, 5G MIMO 3.5GHz.

**Logic flow:**
1. Fig 2: Channel varies slowly → skip is feasible
2. Fig 3: Skipping at noise floor level doesn't hurt NMSE
3. Fig 4: LS delta is practical best feature for scheduling
4. Fig 5: Path diversity determines skip feasibility (not speed alone)

In [ ]:
import sys; sys.path.insert(0, "../..")
import json, numpy as np, matplotlib.pyplot as plt, torch
from pathlib import Path
from src.experiments.pipeline import (
    load_all_ues, _precompute_deltas, _vectorized_scheduling, 
    _compute_nmse_from_tiers, SPEED_LABELS
)
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"font.size": 10, "figure.dpi": 150})

PRESETS = ["munich_elaa_m_1k_15g", "munich_elaa_m_1k_28g", "munich_5g_mimo_3g5"]
PRESET_LABELS = {"munich_elaa_m_1k_15g": "ELAA 15GHz", "munich_elaa_m_1k_28g": "ELAA 28GHz", "munich_5g_mimo_3g5": "5G 3.5GHz"}
MAX_SNAPSHOTS = 1000  # shorter for paper (speed)
UE_PER_SPEED = 2

# Load all 3 configs
all_ue_data = {}
for preset in PRESETS:
    try:
        ud = load_all_ues(preset, max_snapshots=MAX_SNAPSHOTS, ue_per_speed=UE_PER_SPEED)
        all_ue_data[preset] = ud
        print(f"  {PRESET_LABELS[preset]}: {len(ud['ue_list'])} UEs, {ud['max_snapshots']} snaps")
    except Exception as e:
        print(f"  {PRESET_LABELS[preset]}: FAILED ({e})")
        all_ue_data[preset] = None

print(f"\nLoaded: {sum(1 for v in all_ue_data.values() if v is not None)}/3 configs")

## Fig 2: Oracle δ(t) — Channel varies slowly
**Message**: Most slots have δ << 0.1. Skip is fundamentally feasible.
3 rows (static/ped/veh), 3 cols (configs). LS noise floor overlay.

In [ ]:
# Fig 2: Oracle δ time series — 3 speeds × 3 configs
fig, axes = plt.subplots(3, 3, figsize=(14, 8), sharex=True)
dt_ms = 0.25
noise_floor_20dB = np.sqrt(2.0 / 100)  # ≈ 0.141

speed_order = ["static", "ped", "veh"]

for col, preset in enumerate(PRESETS):
    ud = all_ue_data.get(preset)
    if ud is None:
        for row in range(3):
            axes[row, col].text(0.5, 0.5, "No data", ha="center", va="center", transform=axes[row, col].transAxes)
        continue
    
    groups = {"static": [], "ped": [], "veh": []}
    for u in ud["ue_list"]:
        lbl = "static" if u["speed"] < 0.1 else ("ped" if u["speed"] < 5 else "veh")
        groups[lbl].append(u)
    
    for row, spd_label in enumerate(speed_order):
        ax = axes[row, col]
        for u in groups.get(spd_label, []):
            h = u["h_real"]
            diff = h[1:] - h[:-1]
            ref = h[:-1].flatten(1).norm(dim=1).clamp(min=1e-12)
            delta = (diff.flatten(1).norm(dim=1) / ref).numpy()
            t_ms = np.arange(len(delta)) * dt_ms
            ax.plot(t_ms, delta, lw=0.4, alpha=0.7, label=f"UE{u['uid']}")
        
        ax.axhline(noise_floor_20dB, color="orange", ls="--", lw=1, alpha=0.6)
        ax.set_ylim(0, min(0.6, ax.get_ylim()[1] * 1.1) if groups.get(spd_label) else 0.6)
        if row == 0:
            ax.set_title(PRESET_LABELS[preset], fontsize=11)
        if col == 0:
            ax.set_ylabel(f"{spd_label}\nδ_oracle")
        if row == 2:
            ax.set_xlabel("Time (ms)")
        ax.legend(fontsize=6, loc="upper right")

fig.suptitle("Fig 2: Oracle Channel Variation δ(t) across Configs and Mobility\n"
             "(orange dashed = LS noise floor @20dB)", fontsize=12)
plt.tight_layout()
plt.savefig("../../assets/plots/paper_fig2_delta_timeseries.pdf", bbox_inches="tight")
plt.show()

## Fig 3: NMSE vs Threshold — noise floor is safe skip level
**Message**: Oracle scheduling shows that skipping at δ < noise_floor maintains NMSE ≈ full CE for static/ped. Vehicle requires tighter threshold.
3 panels (configs), oracle (blue) vs LS (red), noise floor vertical line.

In [ ]:
# Fig 3: NMSE vs τ — noise floor skip test across all configs
from tqdm.auto import tqdm

SNR_DB = 20.0
noise_floor = np.sqrt(2.0 / (10 ** (SNR_DB / 10)))
thresholds = np.linspace(0.001, 0.3, 25)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
speed_colors = {"static": "tab:blue", "ped": "tab:green", "veh": "tab:red"}

for col, preset in enumerate(PRESETS):
    ax = axes[col]
    ud = all_ue_data.get(preset)
    if ud is None:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        continue
    
    for u in tqdm(ud["ue_list"], desc=f"Fig3 {PRESET_LABELS[preset]}", leave=False):
        h = u["h_real"]
        h_ls, d_ls, d_oracle, _ = _precompute_deltas(h, SNR_DB)
        lbl = "static" if u["speed"] < 0.1 else ("ped" if u["speed"] < 5 else "veh")
        color = speed_colors[lbl]
        
        # Oracle scheduling sweep
        nmses_o = []
        for tau in thresholds:
            sched = _vectorized_scheduling(d_oracle, tau_low=tau, tau_high=2*tau)
            nm = _compute_nmse_from_tiers(h, h_ls, sched["tiers"], alpha=0.7, delta_mode="ema")
            nmses_o.append(10 * np.log10(max(np.mean(nm), 1e-30)))
        
        ax.plot(thresholds, nmses_o, "-", color=color, lw=1.2, alpha=0.7,
                label=f"{lbl}" if u == [x for x in ud["ue_list"] if ("static" if x["speed"]<0.1 else "ped" if x["speed"]<5 else "veh") == lbl][0] else "")
    
    ax.axvline(noise_floor, color="orange", ls=":", lw=2, label=f"NF={noise_floor:.3f}")
    ax.axhline(-10, color="gray", ls=":", alpha=0.5)
    ax.axhline(-20, color="lightgray", ls=":", alpha=0.3)
    ax.set_xlabel("Threshold τ"); ax.set_ylabel("NMSE (dB)")
    ax.set_title(PRESET_LABELS[preset])
    ax.legend(fontsize=7)
    ax.set_ylim(-25, 0)

fig.suptitle("Fig 3: NMSE vs Skip Threshold (Oracle δ scheduling, EMA α=0.7)", fontsize=12)
plt.tight_layout()
plt.savefig("../../assets/plots/paper_fig3_nmse_vs_tau.pdf", bbox_inches="tight")
plt.show()

## Fig 4: Feature Comparison Pareto — SR vs NMSE
**Message**: LS delta is practical best. Oracle delta is upper bound. Decorrelation and power delta are alternatives.
3 panels (configs), 6 feature curves per panel.

In [ ]:
# Fig 4: Feature comparison Pareto front — all configs
def compute_features_cpu(h_real, snr_db=20.0):
    """All features on CPU (safe from OOM)."""
    T = h_real.shape[0]
    sig_pow = h_real.flatten(1).pow(2).mean(1, keepdim=True).sqrt().unsqueeze(-1).unsqueeze(-1)
    h_ls = h_real + (sig_pow / (10 ** (snr_db / 20))) * torch.randn_like(h_real)
    features = {}
    h_flat = h_ls.flatten(1)
    h_flat_o = h_real.flatten(1)
    
    diff = h_flat[1:] - h_flat[:-1]
    features["LS δ"] = (diff.norm(dim=1) / h_flat[:-1].norm(dim=1).clamp(min=1e-12)).numpy()
    diff_o = h_flat_o[1:] - h_flat_o[:-1]
    features["Oracle δ"] = (diff_o.norm(dim=1) / h_flat_o[:-1].norm(dim=1).clamp(min=1e-12)).numpy()
    power = h_flat.pow(2).sum(1)
    features["Power δ"] = (torch.abs(power[1:] - power[:-1]) / power[:-1].clamp(min=1e-12)).numpy()
    dot = (h_flat[1:] * h_flat[:-1]).sum(1)
    norms = h_flat[1:].norm(dim=1) * h_flat[:-1].norm(dim=1)
    features["Decorrelation"] = (1 - dot / norms.clamp(min=1e-12)).numpy()
    return h_ls, features

percentiles = np.linspace(5, 99, 15)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
feat_colors = {"LS δ": "tab:red", "Oracle δ": "tab:blue", "Power δ": "tab:purple", "Decorrelation": "tab:orange"}

for col, preset in enumerate(PRESETS):
    ax = axes[col]
    ud = all_ue_data.get(preset)
    if ud is None:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        continue
    
    feat_results = {}
    for u in tqdm(ud["ue_list"], desc=f"Fig4 {PRESET_LABELS[preset]}", leave=False):
        h = u["h_real"]
        h_ls, features = compute_features_cpu(h, SNR_DB)
        for fn, fv in features.items():
            if fn not in feat_results:
                feat_results[fn] = []
            for tau in np.percentile(fv, percentiles):
                sched = _vectorized_scheduling(fv, tau_low=tau, tau_high=2*tau)
                nm = _compute_nmse_from_tiers(h, h_ls, sched["tiers"], alpha=0.7, delta_mode="ema")
                feat_results[fn].append({
                    "sr": sched["n_skip"]/sched["total"],
                    "nmse": 10*np.log10(max(np.mean(nm), 1e-30))
                })
    
    for fn, pts in feat_results.items():
        sr_bins = {}
        for p in pts:
            sr_bins.setdefault(round(p["sr"], 2), []).append(p["nmse"])
        srs = sorted(sr_bins.keys())
        ax.plot(srs, [np.mean(sr_bins[s]) for s in srs], ".-", ms=3, lw=1.2,
                color=feat_colors.get(fn, "gray"), label=fn, alpha=0.8)
    
    ax.axhline(-10, color="gray", ls=":", alpha=0.5)
    ax.set_xlabel("Skip Rate"); ax.set_ylabel("NMSE (dB)")
    ax.set_title(PRESET_LABELS[preset])
    ax.set_xlim(0, 1); ax.set_ylim(-25, 0)
    if col == 0: ax.legend(fontsize=7)

fig.suptitle("Fig 4: Feature Comparison — Skip Rate vs NMSE (EMA α=0.7)", fontsize=12)
plt.tight_layout()
plt.savefig("../../assets/plots/paper_fig4_feature_pareto.pdf", bbox_inches="tight")
plt.show()

## Fig 5: Path Diversity — why same-speed UEs have different skip feasibility
**Message**: CE skip feasibility depends on multipath richness, not speed alone. LOS-dominant UEs (few paths) are always safe to skip. Multipath-rich UEs require CE even at pedestrian speed.

Uses ELAA 15GHz data (UE3 vs UE4, both ped 1m/s).

In [ ]:
# Fig 5: Path diversity — UE3 (1 path) vs UE4 (118 paths), both ped 1m/s
import h5py

ud = all_ue_data.get("munich_elaa_m_1k_15g")
if ud is not None:
    ue3 = next((u for u in ud["ue_list"] if u["uid"] == 3), None)
    ue4 = next((u for u in ud["ue_list"] if u["uid"] == 4), None)
    
    if ue3 and ue4:
        fig, axes = plt.subplots(2, 2, figsize=(12, 8))
        
        h5_path = str(ud["cfg"].temporal_dir / "channels.h5")
        f = h5py.File(h5_path, "r")
        
        for col, (u, uid, label) in enumerate([(ue3, 3, "UE3 (1 path, LOS)"), (ue4, 4, "UE4 (118 paths, multipath)")]):
            h = u["h_real"]
            
            # Top: CFR power spectrum
            ax = axes[0, col]
            h0 = h[0]
            avg_sc_power = h0.pow(2).sum(0).mean(0).numpy()
            ax.semilogy(avg_sc_power, lw=0.5, color="tab:blue")
            ax.set_xlabel("Subcarrier"); ax.set_ylabel("Power")
            ax.set_title(f"{label}\nCFR Power Spectrum")
            
            # Bottom: NMSE vs skip interval
            ax = axes[1, col]
            h_ls, d_ls, d_oracle, _ = _precompute_deltas(h, 20.0)
            
            skip_intervals = [1, 2, 5, 10, 20, 50, 100, 200, 500]
            T = h.shape[0]
            nmses = []
            for interval in skip_intervals:
                if interval >= T: 
                    nmses.append(float("nan"))
                    continue
                tiers = np.zeros(T, dtype=np.int32)
                tiers[0] = 2
                for t in range(1, T):
                    if t % interval == 0: tiers[t] = 2
                nm = _compute_nmse_from_tiers(h, h_ls, tiers, alpha=0.7, delta_mode="ema")
                nmses.append(10 * np.log10(max(np.mean(nm), 1e-30)))
            
            intervals_ms = [i * 0.25 for i in skip_intervals]
            ax.plot(intervals_ms, nmses, "b-o", ms=4)
            ax.axhline(-10, color="gray", ls=":", alpha=0.5, label="-10dB")
            ax.axhline(-20, color="lightgray", ls=":", alpha=0.3, label="full CE")
            ax.set_xlabel("Skip Interval (ms)"); ax.set_ylabel("NMSE (dB)")
            ax.set_xscale("log")
            ax.set_title(f"NMSE vs Skip Interval")
            ax.legend(fontsize=7)
            
            # Print path info
            n_paths = np.count_nonzero(np.abs(f["cir_a"][0, uid]).sum(axis=(0,1)))
            energy = np.abs(f["cir_a"][0, uid]).sum()
            print(f"{label}: {n_paths} paths, energy={energy:.4e}, median δ={np.median(d_oracle):.6f}")
        
        f.close()
        fig.suptitle("Fig 5: Path Diversity Determines Skip Feasibility\n"
                     "(Both UEs: ped 1 m/s, same config)", fontsize=12)
        plt.tight_layout()
        plt.savefig("../../assets/plots/paper_fig5_path_diversity.pdf", bbox_inches="tight")
        plt.show()
else:
    print("ELAA 15GHz data not loaded")